In [1]:
# Step 1 - Setup
import os
os.chdir('/Users/anshtomar/PROJECTS/Credit-Risk-Scorecard')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_woe = pd.read_csv('data/cs_woe.csv')
print(df_woe.shape)
print("Data loaded!")
print(df_woe.head())


(149999, 8)
Data loaded!
   RevolvingUtilizationOfUnsecuredLines       age  \
0                              1.020577  0.212867   
1                              1.020577  0.277836   
2                              0.297965  0.392068   
3                             -0.688368  0.581351   
4                              1.020577  0.152822   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                              1.901119   0.578974      -0.359045   
1                             -0.257826   0.021594       0.412732   
2                             -0.257826   0.021594       0.412732   
3                             -0.257826   0.021594       0.412732   
4                             -0.257826  -0.230913      -0.422327   

   NumberOfOpenCreditLinesAndLoans  NumberOfDependents  SeriousDlqin2yrs  
0                        -0.041551            0.209356                 1  
1                        -0.046430           -0.088273                 0  
2               

In [2]:
# Train-Test Split:

# step-2 Split data into train and test
from sklearn.model_selection import train_test_split

X = df_woe.drop('SeriousDlqin2yrs', axis=1)
y = df_woe['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (119999, 7)
Test size: (30000, 7)


We split data into 2 parts — 80% to teach the model (train), 20% to test if it learned correctly (test). Like studying from a textbook then giving a surprise test.

In [3]:
# Step 3 - Train Logistic Regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

print("Model trained!")


Model trained!


This is the actual “learning” step. The model looks at all 119,999 training examples and learns the pattern: “when WoE values look like THIS, the person tends to default.”

In [4]:
# Step 4 - Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Predictions done!")
print("Sample predictions:", y_pred[:10])
print("Sample probabilities:", y_pred_proba[:10])


Predictions done!
Sample predictions: [0 0 0 0 0 0 0 0 0 0]
Sample probabilities: [0.00930817 0.0175946  0.02062552 0.0853602  0.13805239 0.04850282
 0.01312102 0.00968149 0.03330144 0.01650328]


Now we test the model on the 30,000 people it has NEVER seen before.

	•	y_pred = Yes/No will they default (0 or 1)
    
	•	y_pred_proba = probability/chance of default (like 0.23 = 23% chance)

    Important note:

Notice person #5 has 13.8% chance — higher than others, meaning slightly riskier customer in this sample.

In [5]:
# Step 5 - Evaluate using AUC
from sklearn.metrics import roc_auc_score

auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")


AUC Score: 0.8127


AUC tells us how good the model is at separating defaulters from non-defaulters.

	•	0.5 = random guessing (bad)
    
	•	0.7-0.8 = good
    
	•	0.8+ = excellent

## 📝 Model Performance — AUC Score

AUC = 0.8127 (Very Good!)

What this means:
- Model can correctly distinguish defaulters 
  from non-defaulters 81% of the time
- This is comparable to industry-standard 
  credit risk models
- WoE transformation clearly worked well

Interview talking point:
"My Logistic Regression model achieved 
0.81 AUC using WoE-transformed features, 
which is in line with industry benchmarks 
for credit scorecards."


In [6]:
# Step 6 - KS Statistic and Gini
from scipy.stats import ks_2samp

ks_stat, _ = ks_2samp(
    y_pred_proba[y_test==1], 
    y_pred_proba[y_test==0]
)

gini = 2 * auc_score - 1

print(f"KS Statistic: {ks_stat:.4f}")
print(f"Gini Coefficient: {gini:.4f}")


KS Statistic: 0.4984
Gini Coefficient: 0.6254


## 📝 Final Model Performance — Industry Comparison

AUC-ROC:  0.8127 (target >0.75) ✅
KS Stat:  0.4984 (target >0.35) ✅
Gini:     0.6254 (target >0.50) ✅

All metrics EXCEED industry benchmarks!

What is KS Statistic?
- Measures max separation between 
  defaulter and non-defaulter distributions
- 0.50 means very strong separation

What is Gini?
- Gini = 2×AUC - 1
- 0.6254 is considered strong for 
  banking/credit risk models

Interview answer:
"My model achieved AUC=0.81, KS=0.50, 
and Gini=0.63 — all exceeding standard 
industry thresholds used at banks like 
Amex for credit risk models."


In [7]:
# Check for overfitting - compare train vs test performance
train_pred_proba = model.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, train_pred_proba)

test_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC:  {test_auc:.4f}")
print(f"Difference: {abs(train_auc - test_auc):.4f}")


Train AUC: 0.8150
Test AUC:  0.8127
Difference: 0.0023


## 📝 Overfitting Check

Train AUC: 0.8150

Test AUC:  0.8127

Difference: 0.0023 (negligible)

Conclusion: NO overfitting detected.

Why?
1. WoE transformation reduces noise — 
   data is already simplified into bins
2. Logistic Regression is a simple model 
   with low variance, unlikely to overfit
3. Train/test difference < 0.02 threshold 
   confirms good generalization

Interview answer:
"I validated against overfitting by 
comparing train (0.815) and test (0.813) 
AUC scores. The negligible 0.002 gap 
confirms the model generalizes well, 
which makes sense given WoE binning 
naturally regularizes the features."
